# 09 – ONNX Export & Dynamic Quantization Optimization

**Purpose:** Export our fine-tuned `DistilBERT` sequence classifier to ONNX format, apply dynamic INT8 quantization, and run comprehensive speed/footprint benchmarks.

This notebook demonstrates:
1. Exporting the PyTorch model to ONNX.
2. Generating dynamically quantized PyTorch and ONNX models.
3. Running inference speed tests comparing standard and optimized versions.
4. Reviewing model size, process RAM utilization, cold start load times, query latency, and throughput (QPS).

## 0. Setup and Environment

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path().resolve().parent
if REPO_ROOT.name != "SupportAI" and (REPO_ROOT / "SupportAI").exists():
    REPO_ROOT = REPO_ROOT / "SupportAI"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

Repo root: C:\Users\gunav\Downloads\SupportAI


## 1. Run Exporter and Benchmarking Suite

We run the ModelOptimizer pipeline to output the ONNX and INT8 files and benchmark their performance.

In [2]:
from src.models.transformer.optimization import ModelOptimizer
from src.utils.constants import OUTPUT_DIR

model_dir = OUTPUT_DIR / "models" / "best_model"

optimizer = ModelOptimizer(model_dir, smoke_run=False)
optimizer.export_onnx()
optimizer.quantize_pytorch()
optimizer.quantize_onnx()
results = optimizer.benchmark_models()

print("Optimization and Benchmarking execution complete!")

[07/13/26 20:42:54] INFO     Exporting model to ONNX format...

C:\Users\gunav\Downloads\SupportAI\src\models\transformer\optimization.py:109: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0713 20:42:55.563000 46280 site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0713 20:42:55.567000 46280 site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0713 20:42:55.569000 46280 site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation f

[torch.onnx] Obtain model graph for `ONNXModelWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ONNXModelWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


[07/13/26 20:42:59] INFO     Removed 2 unused functions

                    INFO     No unused functions to remove

...

[07/13/26 20:46:04] INFO     Benchmarking ONNX Quantized (INT8)...

[07/13/26 20:46:38] INFO     Saving model metrics...

                    INFO     Saving JSON artifact to:                                                              
                             C:\Users\gunav\Downloads\SupportAI\outputs\metrics\optimization_benchmarks.json

                    INFO     Benchmarking suite completed and logged.

Optimization and Benchmarking execution complete!


## 2. Model Optimization Benchmark Results Comparison

We load the generated metrics json file into a DataFrame to easily compare performance.

In [3]:
benchmarks_path = OUTPUT_DIR / "metrics" / "optimization_benchmarks.json"
df = pd.read_json(benchmarks_path).T

print("Model optimization benchmark metrics comparison table:")
df

Model optimization benchmark metrics comparison table:


,Accuracy,Latency (ms),Throughput (QPS),Disk Size (MB),RAM Usage (MB),Cold Start (s)
PyTorch_FP32,0.918587,25.345166,39.455255,255.645061,152.886719,0.054814
PyTorch_INT8,0.910138,14.307380,69.894000,132.344775,265.667969,0.740283
ONNX_FP32,0.918587,25.475250,39.253786,256.430504,262.179688,0.491221
ONNX_INT8,0.917051,12.696822,78.759864,64.809999,70.328125,0.173724


In [4]:
# Export Phase Manifest
import json

from src.api.app import get_git_commit
from src.utils.artifacts import save_yaml

opt_metrics_path = REPO_ROOT / "outputs" / "metrics" / "optimization_benchmarks.json"
opt_metrics = {}
if opt_metrics_path.exists():
    with open(opt_metrics_path) as f:
        opt_metrics = json.load(f)

manifest = {
    "phase": "09_Optimization",
    "optimization_metrics": opt_metrics,
    "git_commit": get_git_commit(),
}
save_yaml(manifest, REPO_ROOT / "outputs" / "manifests" / "phase_09_optimization.yaml")
print("YAML manifest saved successfully:")
print(manifest)


                    INFO     Saving YAML artifact to:                                                              
                             C:\Users\gunav\Downloads\SupportAI\outputs\manifests\phase_09_optimization.yaml

YAML manifest saved successfully:
{'phase': '09_Optimization', 'optimization_metrics': {'PyTorch_FP32': {'Accuracy': 0.9185867895545314, 'Latency (ms)': 25.34516605376173, 'Throughput (QPS)': 39.45525540763147, 'Disk Size (MB)': 255.64506149291992, 'RAM Usage (MB)': 152.88671875, 'Cold Start (s)': 0.05481350002810359}, 'PyTorch_INT8': {'Accuracy': 0.9101382488479263, 'Latency (ms)': 14.307379645163826, 'Throughput (QPS)': 69.8940004949138, 'Disk Size (MB)': 132.34477519989014, 'RAM Usage (MB)': 265.66796875, 'Cold Start (s)': 0.7402828000485897}, 'ONNX_FP32': {'Accuracy': 0.9185867895545314, 'Latency (ms)': 25.475249616443516, 'Throughput (QPS)': 39.253786127949446, 'Disk Size (MB)': 256.43050384521484, 'RAM Usage (MB)': 262.1796875, 'Cold Start (s)': 0.49122129986062646}, 'ONNX_INT8': {'Accuracy': 0.9170506912442397, 'Latency (ms)': 12.69682235200353, 'Throughput (QPS)': 78.75986386800177, 'Disk Size (MB)': 64.80999946594238, 'RAM Usage (MB)': 70.328125, 'Cold Start (s)': 0.1737236001